# Calling Fortran from Python

---
## **Introduction to `f2py`**

[`f2py`](https://numpy.org/doc/stable/f2py/) is a **Fortran-to-Python interface generator** included with NumPy. It allows you to:
- **Wrap Fortran subroutines** and use them in Python.
- **Insert special directives** in Fortran code for complex wrapping.
- **Write interface files** (`.pyf`) to wrap Fortran files without modifying them.

`f2py` automatically generates a `.pyf` template file that can be customized.

---
## **Setup for Fortran Integration**

To work with Fortran in Jupyter Notebook, set up the environment as follows:

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

---
## **Writing Fortran Subroutines**

### **Fortran 90/95 Free Format**

Fortran 90/95 allows for **free-format** code, which is easier to read and write.

**Example: Euclidean Norm Calculation**
```fortran
%%file euclidian_norm.f90
subroutine euclidian_norm(a, b, c)
  real(8), intent(in) :: a, b
  real(8), intent(out) :: c
  c = sqrt(a*a + b*b)
end subroutine euclidian_norm
```

---
### **Fortran 77 Fixed Format**

Fortran 77 uses a **fixed format**, where each line has specific column constraints.

**Example: Euclidean Norm in Fixed Format**
```fortran
%%file euclidian_norm.f
      subroutine euclidian_norm(a, b, c)
      real*8 a, b, c
Cf2py intent(out) c
      c = sqrt(a*a + b*b)
      end
```

---
## **Building Extension Modules with `f2py`**

To compile and use Fortran code in Python, use the `f2py` command:

In [ ]:
%%bash
f2py --quiet -c euclidian_norm.f90 -m vect --fcompiler=gnu95 --f90flags=-O3

**Explanation of Flags:**
- `--quiet`: Suppress verbose output.
- `-c`: Compile the Fortran code.
- `-m vect`: Name the resulting Python module `vect`.
- `--fcompiler=gnu95`: Use the GNU Fortran 95 compiler.
- `--f90flags=-O3`: Enable optimization level 3.

---
## **Using Fortran Subroutines in Python**

After compiling, you can import and use the Fortran subroutine in Python.

**Example: Using the `euclidian_norm` Subroutine**

In [ ]:
import vect
a, b = 3.0, 4.0
c = vect.euclidian_norm(a, b)
print(c)  # Output: 5.0 (Euclidean norm of (3, 4))

---
## **Example: Solving the Laplace Equation with Fortran**

The **Laplace equation** is a second-order partial differential equation used in physics and engineering. Here’s how to solve it using Fortran and Python.

### **Fortran Subroutine for Laplace Equation**
```fortran
%%file laplace_fortran.F90
subroutine laplace_fortran(T, n, residual)
  real(8), intent(inout) :: T(0:n-1, 0:n-1)  ! Python indexing
  integer, intent(in) :: n
  real(8), intent(out) :: residual
  real(8) :: T_old
  integer :: i, j

  residual = 0.0
  do i = 1, n-2
    do j = 1, n-2
      T_old = T(i, j)
      T(i, j) = 0.25 * (T(i+1, j) + T(i-1, j) + T(i, j+1) + T(i, j-1))
      if (T(i, j) > 0) then
        residual = max(residual, abs((T_old - T(i, j)) / T(i, j)))
      end if
    end do
  end do
end subroutine laplace_fortran
```

---
### **Python Code to Call Fortran Subroutine**

**Example: Temperature Distribution**

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import numpy as np
import matplotlib.pyplot as plt

# Boundary conditions
Tnorth, Tsouth, Twest, Teast = 100, 20, 50, 50

# Set meshgrid
n, l = 64, 1.0
X, Y = np.meshgrid(np.linspace(0, l, n), np.linspace(0, l, n))
T = np.zeros((n, n), order='F')  # Fortran-style memory order

# Set boundary conditions
T[n-1:, :] = Tnorth
T[:1, :] = Tsouth
T[:, n-1:] = Teast
T[:, :1] = Twest

# Compile the Fortran code
!f2py --quiet -c laplace_fortran.F90 -m laplace --fcompiler=gnu95 --f90flags=-O3

# Import the compiled module
import laplace

# Iterate to solve the Laplace equation
residual = 1.0
istep = 0
while residual > 1e-5:
    istep += 1
    residual = laplace.laplace_fortran(T, n)
    print((istep, residual), end="\r")

print("\nIterations =", istep)

# Plot the result
plt.rcParams['figure.figsize'] = (10, 6.67)
plt.title("Temperature Distribution")
plt.contourf(X, Y, T)
plt.colorbar()
plt.show()

**Output:** A contour plot of the temperature distribution after solving the Laplace equation.

---
## **References**

For further reading, check out these resources:
- **[Talk by E. Sonnendrücker](http://calcul.math.cnrs.fr/Documents/Journees/dec2006/python-fortran.pdf)**: A talk on integrating Python and Fortran.
- **[SciPy f2py Documentation](https://numpy.org/doc/stable/f2py/)**: Official documentation for `f2py`.
- **[SageMath f2py Documentation](http://www.sagemath.org/doc/numerical_sage/f2py.html)**: Additional `f2py` documentation.
- **Hans Petter Langtangen**, *Python Scripting for Computational Science*, Springer, 2004.

---